In [0]:
-- Game - Faceoffs by Player
-- Faceoff wins and losses for each player in one game, split by zone from that player's side.
-- Change the values in params only; everything below reads from it.
WITH params AS (                              -- change settings here only
  SELECT
    2026020268 AS game_id,          -- e.g. 2025020001 (season 2025, 02 = regular season, game 0001); see Game - Find Game ID
    CAST(NULL AS STRING) AS team    -- team abbreviation, e.g. 'MTL'; NULL = both teams
),
game_players AS (                        -- everyone dressed in the game (game rosters)
  SELECT r.game_id, r.player_id, concat_ws(' ', r.first_name, r.last_name) AS full_name,
         r.position_code, r.sweater_number, r.team_id, t.abbrev AS team,
         r.team_id = g.home_team_id AS is_home, g.home_team_id
  FROM nhl.silver.game_rosters r
  JOIN nhl.gold.dim_game g ON r.game_id = g.game_id
  JOIN nhl.gold.dim_team t ON r.team_id = t.team_id
  CROSS JOIN params pr
  WHERE r.game_id = pr.game_id
),
draws AS (                               -- one row per player per faceoff taken
  SELECT gp.team, gp.is_home, gp.full_name,
         f.winning_player_id = gp.player_id AS won,
         CASE WHEN f.winning_player_id = gp.player_id THEN f.zone_code   -- zone is from the winner's side
              WHEN f.zone_code = 'O' THEN 'D'
              WHEN f.zone_code = 'D' THEN 'O'
              ELSE f.zone_code END AS zone
  FROM nhl.gold.fact_event f
  JOIN game_players gp ON gp.player_id IN (f.winning_player_id, f.losing_player_id)
  CROSS JOIN params pr
  WHERE f.game_id = pr.game_id
    AND f.type_desc_key = 'faceoff'
    AND (pr.team IS NULL OR gp.team = pr.team)
)
SELECT team,
       full_name,
       COUNT_IF(won)                                        AS wins,
       COUNT_IF(NOT won)                                    AS losses,
       ROUND(100.0 * COUNT_IF(won) / COUNT(*), 1)           AS faceoff_pct,
       COUNT_IF(zone = 'O' AND won) || '-' || COUNT_IF(zone = 'O' AND NOT won) AS offensive_zone,
       COUNT_IF(zone = 'N' AND won) || '-' || COUNT_IF(zone = 'N' AND NOT won) AS neutral_zone,
       COUNT_IF(zone = 'D' AND won) || '-' || COUNT_IF(zone = 'D' AND NOT won) AS defensive_zone
FROM draws
GROUP BY team, is_home, full_name
ORDER BY is_home, COUNT(*) DESC;